# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Print top-level name and description
print(f"Dataset name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List the available record sets and their field @ids
record_sets = dataset.metadata.record_sets
print("Available Record Sets (with @id):\n-------------------------------")
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', '')})")
    print("  Fields:")
    for field in rs['fields']:
        print(f"    - {field['@id']} (name: {field.get('name', '')}, dataType: {field.get('dataType', '')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set into a DataFrame
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# For demonstration, let's pick the first record set as main
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Fields in record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets available in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Choose a numeric field for analysis
# Let's automatically select the first numeric field (with dtype 'int' or 'float')
df = dataframes[main_record_set_id]

# Infer a numeric field
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

if numeric_field is not None:
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}):")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to select a categorical/grouping field (the first object-type column)
    group_field = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
            group_field = col
            break

    if group_field:
        # Group by and compute the mean
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Scatter plot if we have a second numeric column
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if len(numeric_fields) > 1:
        plt.figure(figsize=(7, 5))
        sns.scatterplot(x=df[numeric_fields[0]], y=df[numeric_fields[1]])
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.title(f"{numeric_fields[0]} vs. {numeric_fields[1]}")
        plt.show()
else:
    print("Not enough numeric data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* This notebook demonstrated how to use [`mlcroissant`](https://github.com/mlcommons/croissant) to load, inspect, and analyze the FAIR² Mass Spectrometry dataset defined by the Croissant schema at https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json.
* We explored the structure (record sets, fields, their `@id`s) and performed basic data processing and visualization steps.
* For deeper analysis, refer to the specific domain documentation, and use the `@id`s for precise field and record set referencing in your pipelines.
